# 03 — Machine Learning: No-Show Risk Model

Run after Part 1. The test set is held out for final reporting only. Model selection and threshold selection use training data only.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv('/content/outputs/appointments_clean.csv')
X = df[['age','distance_km','fee_inr','lead_days','prior_no_shows']].copy()
X['reminder_sent'] = df['reminder_sent'].map({'Yes': 1, 'No': 0})
X['booking_channel'] = df['booking_channel']
X['visit_type'] = df['visit_type']
X = pd.get_dummies(X, columns=['booking_channel','visit_type'])
y = df['no_show']
appointment_ids = df['appointment_id']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Features:', X.shape[1])
print('Train:', len(X_train), 'Test:', len(X_test), 'Test no-shows:', int(y_test.sum()))
print('Baseline accuracy (always Attended):', round((y_test == 0).mean(), 4))


## Task 3.2 — Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
lr_prob = lr.predict_proba(X_test_s)[:, 1]
lr_pred = lr.predict(X_test_s)
print('Accuracy:', accuracy_score(y_test, lr_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, lr_pred))
print('Precision:', precision_score(y_test, lr_pred))
print('Recall:', recall_score(y_test, lr_pred))
print('F1:', f1_score(y_test, lr_pred))
print('ROC-AUC:', roc_auc_score(y_test, lr_prob))
coef_table = pd.DataFrame({'feature': X.columns, 'coefficient': lr.coef_[0]})
print(coef_table)
reminder_idx = list(X.columns).index('reminder_sent')
print('exp(scaled coefficient):', np.exp(lr.coef_[0][reminder_idx]))
print('exp(No-to-Yes coefficient):', np.exp(lr.coef_[0][reminder_idx] / scaler.scale_[reminder_idx]))


## Task 3.3 — Decision Tree and overfitting

In [ ]:
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)
tree_3 = DecisionTreeClassifier(random_state=42, max_depth=3)
tree_3.fit(X_train, y_train)
for name, model in [('Full tree', tree_full), ('Depth-3 tree', tree_3)]:
    print(name, 'depth=', model.get_depth(),
          'train accuracy=', accuracy_score(y_train, model.predict(X_train)),
          'test accuracy=', accuracy_score(y_test, model.predict(X_test)))
print('Depth-3 feature importance:')
print(pd.Series(tree_3.feature_importances_, index=X.columns).sort_values(ascending=False))


## Task 3.4 — Random Forest and model comparison

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
rf_pred = rf.predict(X_test)
print('RF accuracy:', accuracy_score(y_test, rf_pred))
print('RF recall:', recall_score(y_test, rf_pred))
print('RF ROC-AUC:', roc_auc_score(y_test, rf_prob))
print('Top feature importances:')
print(pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(3))

comparison = pd.DataFrame([
    ['Logistic Regression', accuracy_score(y_test, lr_pred), recall_score(y_test, lr_pred), roc_auc_score(y_test, lr_prob)],
    ['Decision Tree depth 3', accuracy_score(y_test, tree_3.predict(X_test)), recall_score(y_test, tree_3.predict(X_test)), roc_auc_score(y_test, tree_3.predict_proba(X_test)[:,1])],
    ['Random Forest', accuracy_score(y_test, rf_pred), recall_score(y_test, rf_pred), roc_auc_score(y_test, rf_prob)]
], columns=['model','accuracy','recall','roc_auc'])
print(comparison)


## Task 3.5 — Training-only cross-validation, tuning and threshold selection

In [ ]:
lr_pipe = Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
lr_cv = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring='roc_auc').mean()
tree_cv = cross_val_score(DecisionTreeClassifier(random_state=42, max_depth=3), X_train, y_train, cv=5, scoring='roc_auc').mean()
rf_cv = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring='roc_auc').mean()

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {'n_estimators':[100,200], 'max_depth':[3,5,8]},
                    cv=5, scoring='roc_auc')
grid.fit(X_train, y_train)
print('CV AUC — logistic:', lr_cv, 'depth-3 tree:', tree_cv, 'forest:', rf_cv)
print('Tuned forest:', grid.best_params_, grid.best_score_)

# Logistic regression has the highest training CV AUC in the reference run.
winner = lr_pipe
oof_prob = cross_val_predict(winner, X_train, y_train, cv=5, method='predict_proba')[:,1]
for threshold in [0.5, 0.35]:
    flags = (oof_prob >= threshold).astype(int)
    print('Threshold', threshold,
          'precision', precision_score(y_train, flags),
          'recall', recall_score(y_train, flags),
          'F1', f1_score(y_train, flags),
          'flagged', int(flags.sum()))

chosen_threshold = 0.35
winner.fit(X_train, y_train)
test_prob = winner.predict_proba(X_test)[:,1]
test_flag = (test_prob >= chosen_threshold).astype(int)
print('Final test precision:', precision_score(y_test, test_flag))
print('Final test recall:', recall_score(y_test, test_flag))
print('Final test flagged:', int(test_flag.sum()))
print('Final test ROC-AUC:', roc_auc_score(y_test, test_prob))

test_results = pd.DataFrame({
    'appointment_id': appointment_ids.loc[X_test.index].values,
    'no_show': y_test.values,
    'risk_prob': test_prob,
    'flag': test_flag
})
test_results.to_csv('/content/outputs/test_risk_scores.csv', index=False)
print('Saved test_risk_scores.csv')


# Full historical risk list for joining the call-centre notes:
# training rows use out-of-fold probabilities; test rows use held-out probabilities.
all_risk_prob = pd.Series(index=X.index, dtype=float)
all_risk_prob.loc[X_train.index] = oof_prob
all_risk_prob.loc[X_test.index] = test_prob
all_risk_scores = pd.DataFrame({
    'appointment_id': appointment_ids.loc[X.index].values,
    'risk_prob': all_risk_prob.loc[X.index].values,
    'flag': (all_risk_prob.loc[X.index].values >= chosen_threshold).astype(int)
})
all_risk_scores.to_csv('/content/outputs/all_appointment_risk_scores.csv', index=False)
print('Saved all_appointment_risk_scores.csv for the Part 4 join.')


## Interpretation and limitations

The final reference choice is Logistic Regression, selected by training-only cross-validated ROC-AUC, with a 0.35 threshold selected using out-of-fold training predictions. Recall is important because an unflagged no-show leaves a doctor slot unused, while a false positive costs a follow-up call. The threshold should ultimately be set against the front desk's real call capacity.

Limitations: cleaning medians/fences were calculated before splitting; the same patient may appear in train and test; and a random split is not a past-to-future evaluation. The data are observational, so model coefficients are associations, not causal effects.